# Imports & Constants

In [ ]:
import logging
import pickle
import time
from dataclasses import dataclass
from functools import wraps
from pathlib import Path
from typing import Any, Dict, Optional, Tuple, Union

import numpy as np
import optuna
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score, f1_score, log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

In [151]:
# paths
base_path = Path("/kaggle/input/playground-series-s5e12/")
train_csv_path = base_path / "train.csv"
test_csv_path = base_path / "test.csv"
submission_csv_path = base_path / "sample_submission.csv"

# load dataset
train_df = pd.read_csv(train_csv_path)
test_df = pd.read_csv(test_csv_path)

# others
RERUN = False# if true, reruns all visualisations and time-consuming operations
target_col = "diagnosed_diabetes"
features_col = [x for x in train_df.columns if x != target_col and x != "id"]
RANDOM_SEED = 13

# Base Classes

In [152]:
@dataclass
class Args:
    base_path: Path
    objective: str
    features_col: list
    target_col: str

    model_name: str = "xgboost"
    cat_dims: Union[list, None] = None
    cat_idxs: Union[list, None] = None
    one_hot_encode: bool = False
    epochs: int = 2000
    logging_period: int = 200
    use_gpu: bool = True
    early_stopping_rounds: int = 50
    num_splits: int = 5
    random_seed: int = 13
    num_optuna_trials: int = 50

def save_model_to_file(model, args, extension=""):
    args.base_path.mkdir(exist_ok=True, parents=True)
    filename = args.base_path / (str(model.__class__.__name__) + extension + ".pkl")
    pickle.dump(model, open(filename, 'wb'))

def save_predictions_to_file(model, arr, args, extension=""):
    args.base_path.mkdir(exist_ok=True, parents=True)
    filename = args.base_path / (str(model.__class__.__name__) + extension + ".npy")
    np.save(filename, arr)

class BaseModel:
    def __init__(self, params: Dict, args: Any):
        self.params = params
        
        self.model = None

        self.predictions = None
        self.prediction_probabilities = None
    
    def fit(
        self,
        X: np.ndarray,
        y: np.ndarray,
        X_val: Union[None, np.ndarray] = None,
        y_val: Union[None, np.ndarray] = None,
    ) -> Tuple[list, list]:
        self.model.fit(X, y)

        return [], []
    
    def predict(self, X: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        if self.args.objective == "regression":
            self.predictions = self.model.predict(X)
        elif self.args.objective == "classification" or self.args.objective == "binary":
            self.prediction_probabilities = self.predict_proba(X)
            self.predictions = np.argmax(self.prediction_probabilities, axis=1)
        
        return self.predictions

    def save_model_and_predictions(self, y_true: np.ndarray, filename_extension=""):
        self.save_predictions(y_true, filename_extension)
        self.save_model(filename_extension)

    def clone(self):
        return self.__class__(self.params, self.args)

    @classmethod
    def define_trial_parameters(cls, trial: optuna.Trial, args) -> Dict:
        raise NotImplementedError("This method must be implemented by the sub class")

    def save_model(self, filename_extension=""):
        save_model_to_file(self.model, self.args, filename_extension)
    
    def save_predictions(self, y_true: np.ndarray, filename_extension=""):
        if self.args.objective == "regression":
            y = np.concatenate((y_true.reshape(-1, 1).squeeze(), self.predictions.reshape(-1, 1).squeeze()), axis=1)
        else:
            # import pdb; pdb.set_trace()
            y = np.vstack((y_true.reshape(-1, 1).squeeze(), self.predictions)).T
        
        save_predictions_to_file(self.model, y, self.args, filename_extension)


In [ ]:
class XGBoost(BaseModel):
    def __init__(self, params, args):
        super().__init__(params, args)
        self.args = args
        self.params["verbosity"] = 1

        if args.use_gpu:
            self.params["tree_method"] = "hist"
            self.params["device"] = "cuda"
        
        if args.objective == "regression":
            self.params["objective"] = "reg:squarederror"
            self.params["eval_metric"] = "rmse"
        if args.objective == "binary":
            self.params["objective"] = "binary:logistic"
            self.params["eval_metric"] = "auc"
    
    def fit(self, X, y, X_val=None, y_val=None):
        train = xgb.DMatrix(X, label=y)
        val = xgb.DMatrix(X_val, label=y_val)
        eval_list = [(val, "eval")]

        self.model = xgb.train(
            self.params,
            train,
            num_boost_round=self.args.epochs,
            evals=eval_list,
            early_stopping_rounds=self.args.early_stopping_rounds,
            verbose_eval=self.args.logging_period,
        )

        return [], []

    def predict(self, X):
        X = xgb.DMatrix(X)
        return super().predict(X)

    def predict_proba(self, X):
        probabilities = self.model.predict(X)

        if self.args.objective == "binary":
            probabilities = probabilities.reshape(-1, 1).squeeze()
            # import pdb; pdb.set_trace()
            # probabilities = np.concatenate((1 - probabilities, probabilities), 1)
            probabilities = np.vstack((1 - probabilities, probabilities)).T
        
        self.prediction_probabilities = probabilities
        return self.prediction_probabilities

    @classmethod
    def define_trial_parameters(cls, trial):
        params = {
            "max_depth": trial.suggest_int("max_depth", 2, 12, log=True),
            "alpha": trial.suggest_float("alpha", 1e-8, 1.0, log=True),
            "lambda": trial.suggest_float("lambda", 1e-8, 1.0, log=True),
            "eta": trial.suggest_float("eta", 0.01, 0.3, log=True),
        }
        return params

    def save_model(self, filename_extension=""):
        self.args.base_path.mkdir(exist_ok=True, parents=True)
        checkpoint_path = self.args.base_path / (self.__class__.__name__ + filename_extension + ".pkl")
        with open(checkpoint_path, 'wb') as f:
            pickle.dump(self.model, f)

    def load_model(self, checkpoint_path):
        with open(checkpoint_path, 'rb') as f:
            self.model = pickle.load(f)
        return self

# Utils

In [ ]:


def time_it(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        print(f"{func.__name__} took {end - start:.4f} seconds")
        return result
    return wrapper

def load_data(args: Args, df: pd.DataFrame) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    X = df[features_col].to_numpy()
    if target_col in df.columns:
        y = df[target_col].to_numpy()
    else:
        y = None

    # preprocess
    non_cat_idxs = []
    for i in range(len(args.features_col)):
        if args.cat_idxs and i in args.cat_idxs:
            # categorical feature
            le = LabelEncoder()
            X[:, i] = le.fit_transform(X[:, i])

            args.cat_dims.append(len(le.classes_))
        else:
            non_cat_idxs.append(i)

    if args.one_hot_encode:
        ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
        new_x1 = ohe.fit_tranform(X[:, args.cat_idxs])
        new_x2 = X[:, non_cat_idxs]

        X = np.concatenate((new_x1, new_x2), axis=1)
    
    return X, y

class Scorer:
    def eval(y_true: np.ndarray, y_pred: np.ndarray, y_probabilities:np.ndarray):
        raise NotImplementedError("This method must be implemented by the sub class")
    
    def get_results(self):
        raise NotImplementedError("This method must be implemented by the sub class")
    
    def get_objective_result(self):
        raise NotImplementedError("This method must be implemented by the sub class")


class BinScorer(Scorer):
    def __init__(self):
        self.loglosses = []
        self.aucs = []
        self.accs = []
        self.f1s = []
    
    def eval(self, y_true, y_pred, y_probabilities):
        logloss = log_loss(y_true, y_probabilities[:, 1])
        auc = roc_auc_score(y_true, y_probabilities[:, 1])
        acc = accuracy_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred)

        self.loglosses.append(logloss)
        self.aucs.append(auc)
        self.accs.append(acc)
        self.f1s.append(f1)
    
    def get_results(self):
        logloss_mean = np.mean(self.loglosses)
        logloss_std = np.std(self.loglosses)

        auc_mean = np.mean(self.aucs)
        auc_std = np.std(self.aucs)

        acc_mean = np.mean(self.accs)
        acc_std = np.std(self.accs)

        f1_mean = np.mean(self.f1s)
        f1_std = np.std(self.f1s)

        return {
            "logloss_mean": logloss_mean,
            "logloss_std": logloss_std,
            "auc_mean": auc_mean,
            "auc_std": auc_std,
            "acc_mean": acc_mean,
            "acc_std": acc_std,
            "f1_mean": f1_mean,
            "f1_std": f1_std,
        }

    def get_objective_result(self):
        return np.mean(self.aucs)

# Training Utils

In [ ]:

class Objective(object):
    def __init__(self, args, model, X, y):
        self.model = model
        self.args = args
        self.X = X
        self.y = y
    
    def __call__(self, trial: optuna.Trial):
        trial_params = self.model.define_trial_parameters(trial)
        model = self.model(trial_params, self.args)

        sc = cross_validation(model, self.X, self.y, self.args)

        # save_hyperparameters(self.args, trial_params, sc.get_results())

        return sc.get_objective_result()

@time_it
def cross_validation(model, X, y, args, save_models: bool = False) -> Scorer:
    scorer = BinScorer()

    if args.objective == "binary":
        skf = StratifiedKFold(n_splits=args.num_splits, shuffle=True, random_state=args.random_seed)
    
    for idx, (train_index, val_index) in enumerate(skf.split(X, y)):
        X_train, X_val = X[train_index], X[val_index]
        y_train, y_val = y[train_index], y[val_index]

        model_fold = model.clone()
        loss_history, val_loss_history = model_fold.fit(X_train, y_train, X_val, y_val)

        model_fold.predict(X_val)
        y_pred = model_fold.predictions
        y_prob = model_fold.prediction_probabilities

        if save_models:
            model_fold.save_model_and_predictions(y_val, f"_fold{idx}")

        scorer.eval(y_val, y_pred, y_prob)
        # display(scorer.get_results())
    
    return scorer

@time_it
def train(args) -> Dict[str, Any]:
    print("start hyperparemeter optimization")
    X, y = load_data(args, train_df)

    args.base_path.mkdir(exist_ok=True, parents=True)

    optuna.logging.get_logger("optuna").addHandler(logging.StreamHandler())
    # optuna.logging.set_verbosity(optuna.logging.ERROR)

    study_name = args.model_name + "_study"
    storage_name = "sqlite:///" + str(args.base_path / (study_name + ".db"))
    
    study = optuna.create_study(
        direction="minimize",
        study_name=study_name,
        storage=storage_name,
        load_if_exists=True,
    )
    study.optimize(Objective(args, XGBoost, X, y), n_trials=args.num_optuna_trials)
    print("Best Parameters: ", study.best_trial.params)

    model = XGBoost(study.best_trial.params, args)
    cross_validation(model, X, y, args, save_models=True)

    return study.best_trial.params

# Run

In [ ]:
args = Args(
    base_path=Path("ign_training_xgboost"),
    objective="binary",
    features_col=features_col,
    target_col=target_col,
    cat_idxs=[15, 16, 17, 18, 19, 20],
    cat_dims=[],
    # epochs=2,
    # logging_period=1,
    # early_stopping_rounds=2,
    # num_optuna_trials=5,
)
best_study_params = train(args)

start hyperparemeter optimization
[0]	eval-auc:0.67987
[1]	eval-auc:0.68240
[0]	eval-auc:0.68079
[1]	eval-auc:0.68258
[0]	eval-auc:0.68348
[1]	eval-auc:0.68495
[0]	eval-auc:0.68055
[1]	eval-auc:0.68235
[0]	eval-auc:0.68131
[1]	eval-auc:0.68323
cross_validation took 2.4872 seconds
[0]	eval-auc:0.67987
[1]	eval-auc:0.68272
[0]	eval-auc:0.68079
[1]	eval-auc:0.68258
[0]	eval-auc:0.68348
[1]	eval-auc:0.68493
[0]	eval-auc:0.68055
[1]	eval-auc:0.68240
[0]	eval-auc:0.68131
[1]	eval-auc:0.68331
cross_validation took 2.5047 seconds
[0]	eval-auc:0.67987
[1]	eval-auc:0.68272
[0]	eval-auc:0.68079
[1]	eval-auc:0.68258
[0]	eval-auc:0.68348
[1]	eval-auc:0.68526
[0]	eval-auc:0.68055
[1]	eval-auc:0.68240
[0]	eval-auc:0.68131
[1]	eval-auc:0.68332
cross_validation took 2.5101 seconds
[0]	eval-auc:0.67987
[1]	eval-auc:0.68272
[0]	eval-auc:0.68079
[1]	eval-auc:0.68258
[0]	eval-auc:0.68348
[1]	eval-auc:0.68493
[0]	eval-auc:0.68055
[1]	eval-auc:0.68240
[0]	eval-auc:0.68131
[1]	eval-auc:0.68332
cross_validatio

# Use ensemble for final prediction

In [157]:
all_predictions = []
X_test, y_test = load_data(args, test_df)
for fold in range(args.num_splits):
    # load model from pkl
    curr_model_path = f"XGBoost_fold{fold}.pkl"
    model = XGBoost(best_study_params, args)
    model.load_model(args.base_path / curr_model_path)

    model.predict(X_test)
    curr_predictions = model.prediction_probabilities[:, 1].tolist()
    all_predictions.append(curr_predictions)

all_predictions = np.array(all_predictions)
final_predictions = np.mean(all_predictions, axis=0)

# Submission

In [158]:
sub_df = pd.read_csv(submission_csv_path)
sub_df[target_col] = final_predictions
sub_df.to_csv('submission.csv', index=False)